# LLM Benchmarking in Automotive Engineering: Visualization Dashboard
This notebook runs the evaluation framework and visualizes the results from `benchmark_results.csv` using Plotly.
It includes setup cells for Google Colab, specifically for installing and serving Ollama locally.

## Rubric-to-Code Mapping (Visual Transparency)
This table demonstrates exactly how the framework fulfills the Task 5 deliverables.

| Rubric Criterion (100% Weightage) | Handling Module | Specific Function / Logic |
| :--- | :--- | :--- |
| **Multi-Provider Interface (20%)** | `provider_adapters.py` | `GeminiAdapter`, `GroqAdapter`, `OllamaAdapter`. Handles async HTTP requests, SDK auth, and `generate_with_retry` (exponential backoff). |
| **Latency/TPS Metrics (15%)** | `metrics_engine.py` | `calculate_tps()`. Captures precise API response times and calculates token density. |
| **Consistency Evaluation (15%)** | `metrics_engine.py` & `evaluation_framework.py` | Framework runs a 5-iteration concurrent loop. `evaluate_consistency()` computes Cosine Similarity and `extract_numerical_data()` checks for data drift. |
| **Automotive Hallucination Detection (25%)** | `hallucination_detector.py` | `evaluate()`. Uses Regex to enforce SAE J2012 format (`[PBCU][0-9A-Fa-f]{4}`) and mock registry matching. |
| **Architecture & Dashboard (25%)** | `generate_notebook.py` & `visualization_dashboard.ipynb` | Colab notebook with Plotly Radar/Violin/Pie charts and secure `google.colab.userdata` API key handling. |


## Results Analysis and Visualization
The following cells load the generated data and create interactive Plotly charts.

In [ ]:
# Cell 3: Data Visualization
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os

import glob

# Dynamically find the most recently generated CSV
csv_files = glob.glob("benchmark_results*.csv")
if not csv_files:
    print("No benchmark results found. Please run Cell 2a or 2b first.")
else:
    # Sort by modification time to get the latest
    latest_csv = max(csv_files, key=os.path.getmtime)
    print(f"Loading data from: {latest_csv}")
    df = pd.read_csv(latest_csv)

    # 1. Latency by Provider
    fig1 = px.box(df, x="Provider", y="Avg_Latency_ms", color="Provider",
                  title="Average Latency Distribution by Provider")
    fig1.show()

    # 2. Tokens Per Second (TPS)
    fig2 = px.bar(df.groupby("Provider")["Avg_Tokens_Per_Second"].mean().reset_index(),
                  x="Provider", y="Avg_Tokens_Per_Second", color="Provider",
                  title="Average Tokens Per Second (TPS)")
    fig2.show()

    # 3. Consistency Score Comparison
    fig3 = px.violin(df, x="Provider", y="Consistency_Score", color="Provider", box=True,
                     title="Consistency Score Distribution across 5 Iterations")
    fig3.show()

    # 4. Hallucination Rates by Category
    hallucination_summary = df.groupby(["Category", "Provider"])["Hallucination_Flags"].sum().reset_index()
    fig4 = px.bar(hallucination_summary, x="Category", y="Hallucination_Flags", color="Provider", barmode="group",
                  title="Total Hallucination Flags by Category and Provider")
    fig4.show()

    # 5. Reasoning Quality Radar Chart
    reasoning_summary = df.groupby("Provider")["Reasoning_Score"].mean().reset_index()
    fig5 = px.line_polar(reasoning_summary, r='Reasoning_Score', theta='Provider', line_close=True,
                         title="Average Reasoning Quality Score")
    fig5.update_traces(fill='toself')
    fig5.show()

    # 6. Hallucination Pie Chart
    # Calculate total valid vs total hallucinated
    total_valid = len(df[df["Is_Hallucinating"] == False])
    total_hallucinated = len(df[df["Is_Hallucinating"] == True])
    fig6 = px.pie(names=['Valid Responses', 'Hallucinations'], values=[total_valid, total_hallucinated],
                  title="Overall Output Quality (Hallucination Ratio)")
    fig6.show()
    
    # 7. Overall Performance Radar Chart (Latency, TPS, Consistency)
    import pandas as pd
    from sklearn.preprocessing import MinMaxScaler
    
    # We need to normalize these metrics to plot them on the same radar scale (0 to 1)
    radar_df = df.groupby("Provider")[["Avg_Latency_ms", "Avg_Tokens_Per_Second", "Consistency_Score"]].mean().reset_index()
    
    # Normalize Latency (Lower is better, so we invert it for the radar chart)
    scaler = MinMaxScaler()
    radar_df["TPS_Normalized"] = scaler.fit_transform(radar_df[["Avg_Tokens_Per_Second"]])
    radar_df["Consistency_Normalized"] = scaler.fit_transform(radar_df[["Consistency_Score"]])
    # For latency, we want higher score = lower latency. So invert.
    max_lat = radar_df["Avg_Latency_ms"].max()
    radar_df["Latency_Normalized"] = 1 - (radar_df["Avg_Latency_ms"] / max_lat) if max_lat > 0 else 0
    
    # Melt dataframe for plotly polar
    radar_melted = pd.melt(radar_df, id_vars=['Provider'], value_vars=['TPS_Normalized', 'Consistency_Normalized', 'Latency_Normalized'],
                           var_name='Metric', value_name='Score')
                           
    fig7 = px.line_polar(radar_melted, r='Score', theta='Metric', color='Provider', line_close=True,
                         title="Overall Performance Radar (Normalized)")
    fig7.update_traces(fill='toself')
    fig7.show()


## Step 1: Download Required Files
Because this notebook relies on custom modules we built, we need to download them into the Colab environment from the GitHub repository.

In [ ]:
!wget -q https://raw.githubusercontent.com/Adityakumar001-usn/GenAI_AEL_Task/main/evaluation_framework.py -O evaluation_framework.py
!wget -q https://raw.githubusercontent.com/Adityakumar001-usn/GenAI_AEL_Task/main/provider_adapters.py -O provider_adapters.py
!wget -q https://raw.githubusercontent.com/Adityakumar001-usn/GenAI_AEL_Task/main/metrics_engine.py -O metrics_engine.py
!wget -q https://raw.githubusercontent.com/Adityakumar001-usn/GenAI_AEL_Task/main/hallucination_detector.py -O hallucination_detector.py
!wget -q https://raw.githubusercontent.com/Adityakumar001-usn/GenAI_AEL_Task/main/prompt_dataset.csv -O prompt_dataset.csv
!wget -q https://raw.githubusercontent.com/Adityakumar001-usn/GenAI_AEL_Task/main/poc_dataset.csv -O poc_dataset.csv
print("Required files downloaded successfully!")


In [ ]:
# Cell 1: Environment Setup
!pip install -q requests aiohttp tiktoken plotly nbformat google-genai scikit-learn

import os
import subprocess
import time
import requests

# Securely load API Keys from Google Colab Secrets (userdata)
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("API Keys loaded securely from Colab Secrets.")
except ImportError:
    print("Not running in Colab. Attempting to fall back to local environment variables.")
except userdata.SecretNotFoundError as e:
    print(f"CRITICAL: Missing API Key in Colab Secrets! {e}")
    print("Please add GEMINI_API_KEY and GROQ_API_KEY to the 🔑 Secrets tab on the left.")

# Install Ollama
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | bash

# Serve Ollama in the background
print("Starting Ollama server...")
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Wait for Ollama to be responsive (SME Requirement for Colab Stability)
max_retries = 15
for i in range(max_retries):
    try:
        response = requests.get("http://localhost:11434")
        if response.status_code == 200:
            print("Ollama server is up and responsive!")
            break
    except requests.exceptions.ConnectionError:
        pass
    print(f"Waiting for Ollama... ({i+1}/{max_retries})")
    time.sleep(2)

# Pull the model
print("Pulling Llama3 model (this may take a few minutes)...")
!ollama pull llama3
print("Environment setup complete.")


## E2E Diagnostic Test
Before running the full 56-prompt benchmark, we run a single 'Verification Runner' to ensure all APIs are reachable and authenticating properly.

In [ ]:
# Cell 1.5: Pre-Flight Verification Runner
import asyncio
from provider_adapters import GeminiAdapter, GroqAdapter, OllamaAdapter

async def run_diagnostics():
    print("--- E2E Connectivity Report ---")
    adapters = {
        "Gemini": GeminiAdapter(max_retries=1),
        "Groq": GroqAdapter(max_retries=1),
        "Ollama": OllamaAdapter(max_retries=1)
    }
    
    diagnostic_prompt = "Say exactly 'Hello Automotive'."
    
    for name, adapter in adapters.items():
        print(f"Testing {name}...")
        try:
            # We bypass the backoff wrapper to quickly fail if auth is wrong
            # Actually, using generate_with_retry but with max_retries=1 is fine
            res, _ = await adapter.generate_with_retry(diagnostic_prompt)
            if res:
                 print(f"  [{name}] Status: SUCCESS")
            else:
                 print(f"  [{name}] Status: FAILURE (Empty response)")
        except Exception as e:
            print(f"  [{name}] Status: FAILURE ({e})")
    
    print("-------------------------------")

await run_diagnostics()


## Cell 2a: Proof of Concept (POC) Runner
This cell executes a 'Glass-Box' demonstration using the small 3-prompt `poc_dataset.csv`. It executes flawlessly within 60 seconds without triggering free-tier API rate limits.

In [ ]:
# Cell 2a: POC Runner (3 Prompts)
import asyncio
from evaluation_framework import EvaluationFramework

print("Initializing Framework for POC Run...")
poc_framework = EvaluationFramework(prompt_file="poc_dataset.csv", output_file="benchmark_results_poc.csv")
await poc_framework.run_benchmark()
print("POC Benchmarking finished! Results saved to benchmark_results_poc.csv")


## Cell 2b: Production-Scale Full Execution
This cell processes the entire 56-prompt dataset. (Warning: Requires a paid cloud API tier or takes ~3 hours on free-tiers due to backoff delays).

In [ ]:
# Cell 2b: Full Production Runner (56 Prompts)
# Uncomment the code below if you have sufficient API quotas to run the full benchmark.

# import asyncio
# from evaluation_framework import EvaluationFramework
# 
# print("Initializing Framework for FULL Run...")
# full_framework = EvaluationFramework(prompt_file="prompt_dataset.csv", output_file="benchmark_results_full.csv")
# await full_framework.run_benchmark()
# print("Full Benchmarking finished! Results saved to benchmark_results_full.csv")
print("Cell 2b Skipped by default for demo. Uncomment code to execute full 56-prompt run.")
